<a href="https://colab.research.google.com/github/goutham3010/Hospital-Readmission-Prediction-System/blob/main/06_hyperparameter_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ============================================
# STEP 06: HYPERPARAMETER TUNING
# Hospital Readmission Prediction
# ============================================

In [ ]:
# ============================================
# 1. Import Required Libraries
# ============================================
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# ============================================
# 2. Define Hyperparameter Grid
# ============================================
# Note: 'model__' prefix is used if random_forest_model is a Pipeline containing 'model'
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

In [ ]:
# ============================================
# 3. Function: Perform Hyperparameter Tuning
# ============================================
def tune_hyperparameters(estimator, param_grid, X_train, y_train, cv=5, scoring="f1"):
    """
    Performs Grid Search Cross-Validation to find the optimal hyperparameters.
    """
    print("Starting Grid Search Cross-Validation...")
    grid_search = GridSearchCV(
        estimator=estimator,
        param_grid=param_grid,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    print("\n" + "=" * 60)
    print("BEST HYPERPARAMETERS")
    print("=" * 60)
    for param, value in grid_search.best_params_.items():
        print(f"{param}: {value}")

    print("\n" + "=" * 60)
    print("BEST TUNED MODEL")
    print("=" * 60)
    print(grid_search.best_estimator_)

    return grid_search, grid_search.best_estimator_

# Execute Grid Search
grid_search, best_rf_model = tune_hyperparameters(random_forest_model, param_grid, X_train, y_train)

In [ ]:
# ============================================
# 4. Function: Evaluate Tuned Model Metrics
# ============================================
def evaluate_tuned_model(model, X_test, y_test, model_name="Tuned Random Forest"):
    """
    Calculates and prints evaluation metrics for the tuned model.
    """
    y_pred = model.predict(X_test)

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0)
    }

    print("=" * 60)
    print(f"{model_name.upper()} PERFORMANCE")
    print("=" * 60)
    for k, v in metrics.items():
        print(f"{k:<10}: {v:.4f}")

    return y_pred, metrics

# Evaluate Tuned Model
y_pred_tuned, tuned_metrics = evaluate_tuned_model(best_rf_model, X_test, y_test)

In [ ]:
# ============================================
# 5. Function: Display Detailed Classification Report
# ============================================
def display_classification_report(y_test, y_pred, title="Tuned Random Forest"):
    """
    Prints a formatted classification report.
    """
    print("=" * 60)
    print(f"CLASSIFICATION REPORT: {title}")
    print("=" * 60)
    print(classification_report(
        y_test,
        y_pred,
        target_names=["Not Readmitted (0)", "Readmitted (1)"],
        zero_division=0
    ))

# Print Classification Report
display_classification_report(y_test, y_pred_tuned)

In [ ]:
# ============================================
# 6. Function: Plot Confusion Matrix
# ============================================
def plot_confusion_matrix(y_test, y_pred, title="Confusion Matrix - Tuned Random Forest"):
    """
    Plots a styled confusion matrix.
    """
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Not Readmitted (0)", "Readmitted (1)"]
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title(title, fontsize=13, fontweight="bold", pad=12)
    plt.tight_layout()
    plt.show()

# Plot Confusion Matrix
plot_confusion_matrix(y_test, y_pred_tuned)

In [ ]:
# ============================================
# 7. Function: Compare Original vs Tuned Model
# ============================================
def compare_original_vs_tuned(original_model, tuned_model, X_test, y_test):
    """
    Compares metrics between the original and hyperparameter-tuned model.
    """
    y_pred_orig = original_model.predict(X_test)
    y_pred_tune = tuned_model.predict(X_test)

    comparison_data = [
        {
            "Model": "Original Random Forest",
            "Accuracy": round(accuracy_score(y_test, y_pred_orig), 4),
            "Precision": round(precision_score(y_test, y_pred_orig, zero_division=0), 4),
            "Recall": round(recall_score(y_test, y_pred_orig, zero_division=0), 4),
            "F1 Score": round(f1_score(y_test, y_pred_orig, zero_division=0), 4)
        },
        {
            "Model": "Tuned Random Forest",
            "Accuracy": round(accuracy_score(y_test, y_pred_tune), 4),
            "Precision": round(precision_score(y_test, y_pred_tune, zero_division=0), 4),
            "Recall": round(recall_score(y_test, y_pred_tune, zero_division=0), 4),
            "F1 Score": round(f1_score(y_test, y_pred_tune, zero_division=0), 4)
        }
    ]

    comp_df = pd.DataFrame(comparison_data)

    print("=" * 60)
    print("ORIGINAL VS TUNED MODEL COMPARISON")
    print("=" * 60)
    display(comp_df)

    return comp_df

# Run Comparison
comparison_df = compare_original_vs_tuned(random_forest_model, best_rf_model, X_test, y_test)

In [ ]:
# ============================================
# 8. Final Model Assignment & Summary
# ============================================
final_model = best_rf_model

print("=" * 60)
print("STEP 06 COMPLETED: FINAL MODEL READY")
print("=" * 60)
print("Final Model Pipeline:")
print(final_model)

print("\nBest Parameters:")
print(grid_search.best_params_)